## 0. 준비

- 1일차 도구(`requests`, `BeautifulSoup`)로 먼저 시작

In [19]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup

user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
headers = {"User-Agent": user_agent}

---
## 1. 다음 영화순위를 직접 가져와보기

- 대상: 다음 영화순위 검색 페이지
- 1일차에 배운 `requests`로 페이지를 가져와 HTML 파일로 저장

In [29]:
url = "https://search.daum.net/search?w=tot&q=2025%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"

# headers는 키워드 인자로 전달
response = requests.get(url, headers=headers)
response.raise_for_status()

# 받은 HTML을 파일로 저장
#with open("mymovie.html", "w", encoding="utf-8") as fd:
#    fd.write(response.text)

print("저장 완료. HTML 길이:", len(response.text))

저장 완료. HTML 길이: 260586


### 저장한 HTML을 브라우저로 열어보기

- `mymovie.html`을 브라우저로 직접 열어보면 **영화 목록이 안 보임**
- 그런데 실제 다음 사이트를 브라우저로 열면 영화 목록이 **잘 보임**
- 같은 페이지인데 저장본만 비어 있음 → 뭔가 이상함

### 원인

- 다음 영화순위는 페이지를 연 뒤 **자바스크립트가 나중에 목록을 그림**
- `requests`는 서버가 처음 준 **HTML(자바스크립트 실행 전)**만 받음 → 목록이 비어 있음
- 이런 **동적 페이지**는 `requests`로 부족 → 브라우저를 직접 띄우는 **Selenium** 필요

---
## 2. Selenium 개요

- 브라우저를 코드로 자동 조작하는 도구
- 쓰는 경우: 자바스크립트 동적 데이터 수집, 클릭/입력/스크롤 이벤트

### 셀레니움 · 드라이버 · 브라우저 관계

```text
 [내 파이썬 코드]            [드라이버]               [크롬 브라우저]
  selenium 명령    ──▶    chromedriver    ──▶    실제 크롬 창
  driver.get(...)         (명령 통역사)           페이지 열기/클릭/입력
  find_element()
         ◀────  page_source(자바스크립트 실행 결과)  ────◀
```

- 드라이버가 진짜 크롬을 움직이고, **자바스크립트 실행 결과**를 `page_source`로 돌려줌
- 그래서 `requests`와 달리 동적 데이터까지 받을 수 있음

### 설치와 드라이버

```text
pip install selenium  (1회, 직접 설치)
        ↓
webdriver.Chrome() 첫 호출 → Selenium Manager가 드라이버 자동 처리
        ↓
크롬 창 뜨고 동작
```

- 드라이버(chromedriver)는 **자동** 처리됨 (Selenium 4.6+). 수동 다운로드 불필요
- 단, **크롬 브라우저 본체는 설치**돼 있어야 하고, 첫 실행 시 **인터넷 연결** 필요
- `user_agent`(0번에서 만든 것)는 뒤 Selenium 옵션(ChromeOptions)에서도 사용

In [ ]:
# 셀레늄 설치 # 한번만 하면 됨

# pip install selenium

In [30]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time
from io import StringIO

print("selenium 버전:", selenium.__version__)

selenium 버전: 4.45.0


---
## 3. 브라우저 기본 조작

- `webdriver.Chrome()` : 크롬 실행
- `get(url)` : 페이지 이동
- `back()`, `forward()`, `refresh()` : 뒤로 / 앞으로 / 새로고침
- `quit()` : 종료

In [4]:
# 셀레늄을 이용해 크롬 실행

driver = webdriver.Chrome()


In [6]:
# 네이버 접속

driver.get("https://naver.com")
time.sleep(1)                   # 여러 코드가 있으면 잠시 멈춰줌

In [8]:
# 새로고침

driver.refresh()
time.sleep(1)

In [9]:
# 끄기

driver.quit()

In [10]:
# with 구문: 블록이 끝나면 자동 종료 (quit 빠뜨림 방지)
with webdriver.Chrome() as driver:
    driver.get("https://naver.com")
    time.sleep(1)
    print("페이지 제목:", driver.title)

페이지 제목: NAVER


In [14]:
# By.TAG_NAME : 모든 a(링크) 찾기
with webdriver.Chrome() as driver:
    driver.get("https://naver.com")
    time.sleep(5)
    links = driver.find_elements(By.TAG_NAME, "a")   # a태그를 찾아서 link에 넣기
    print("링크 개수:", len(links))                   # 링크의 개수 표시
    for a in links[:5]:                              # a태그의 링크주소 출력
        print(a.get_attribute("href"))

링크 개수: 160
https://www.naver.com/#topAsideButton
https://www.naver.com/#shortcutArea
https://www.naver.com/#newsstand
https://www.naver.com/#shopping
https://www.naver.com/#feed


In [15]:
# By.NAME + send_keys(입력) + Keys.ENTER(엔터)
with webdriver.Chrome() as driver:
    driver.get("https://naver.com")
    time.sleep(3)

    box = driver.find_element(By.NAME, "query")   # 검색창
    box.send_keys("파이썬")                        # 입력
    time.sleep(1)
    box.send_keys(Keys.ENTER)                      # 검색(엔터키 입력)
    time.sleep(2)
    print("검색 후 URL:", driver.current_url)

검색 후 URL: https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&ackey=bql17z9m


In [16]:
# 같은 검색창을 By.ID / By.XPATH / By.CSS_SELECTOR 세 방법으로 찾기
with webdriver.Chrome() as driver:
    driver.get("https://naver.com")
    time.sleep(1)

    box1 = driver.find_element(By.ID, "query")                 # id
    box2 = driver.find_element(By.XPATH, '//*[@id="query"]')   # xpath
    box3 = driver.find_element(By.CSS_SELECTOR, "#query")      # css 선택자

    # 셋 다 name 속성이 'query'면 같은 검색창을 찾은 것
    print("id로 찾을 경우 :", box1.get_attribute("name"))
    print("xpath로 찾을 경우 :", box2.get_attribute("name"))
    print("css로 찾을 경우 :", box3.get_attribute("name"))

id로  : query
xpath로: query
css로 : query


In [31]:
# By.LINK_TEXT : 링크 글자로 클릭
with webdriver.Chrome() as driver:
    driver.get("https://naver.com")
    time.sleep(1)
    driver.find_element(By.LINK_TEXT, "지도").click()  # 지도 글자에 링크가 걸려있는 요소 찾기
    time.sleep(2)

    # "지도"는 새 탭에서 열림. driver는 아직 첫 탭을 보고 있어 current_url이 naver임
    print("탭 개수 :", len(driver.window_handles))          # 2 -> 새 탭 열림
    driver.switch_to.window(driver.window_handles[-1])     # 마지막(새) 탭으로 전환
    print("이동한 URL :", driver.current_url)               # 이제 지도 주소

탭 개수 : 2
이동한 URL : https://map.naver.com/p/


---
## 5. 대기 (Waits)

- `time.sleep(2)` : 무조건 2초 (간단하지만 비효율)
- `WebDriverWait + EC` : 조건 만족까지만 대기 (동적 페이지에 권장)

```python
WebDriverWait(driver, 10).until(           # driver를 최대 10초까지 지켜보다가
    EC.presence_of_element_located(         # 요소가 화면에 나타나면
        (By.CSS_SELECTOR, "div.item-thumb c-thumb img") # 요소 = 실제 수집 대상 img
    )
)                                           # 나타나면 바로 다음 줄로 진행
```

- 10초는 최대 한도일 뿐, 조건이 만족되면 바로 넘어감
- 기다릴 요소는 **실제로 가져올 요소(img)**로 지정 (그래야 다 그려진 뒤 진행)

---
## 6. 실전 1: 다음 영화 썸네일 수집

- 1번에서 `requests`로 못 가져온 그 페이지를 Selenium으로 해결
- 페이지 구조 (자바스크립트 실행 후 기준):

```text
div.item-thumb              영화 한 편의 썸네일 영역
  └─ c-thumb                썸네일 커스텀 요소 (JS로 나중에 채워짐)
       └─ div
            └─ a            작품 링크
                 └─ img     포스터 이미지 (src = 이미지 주소)  ← 수집 목표
```

- 안쪽 `img`는 JS로 나중에 채워짐 → 대기는 `img` 기준 (5번 참고)

In [20]:
url = "https://search.daum.net/search?w=tot&q=2025%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"

# user-agent 옵션 (봇 차단 대비)
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)
driver.get(url)

# img가 나타날 때까지 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item-thumb c-thumb img"))
)

soup = BeautifulSoup(driver.page_source, "lxml")
images = soup.select("div > div.item-thumb > c-thumb > div > a > img")

# plazy(지연 로딩) 제외
image_urls = [img["src"] for img in images if "plazy" not in img["src"]]
print("전체 img 수:", len(images), "/ 실제 썸네일 수:", len(image_urls))

driver.quit()

전체 img 수: 33 / 실제 썸네일 수: 10


### 1단계: 다음 버튼 클릭해보기 (11~20위 열기)

- 한 화면엔 10위까지만 보임 → **다음(>) 버튼**을 누르면 다음 순위가 뜸
- 다음 버튼 CSS 선택자: `#mor_history_id_0 > div > div.compo-paging > button.btn_next`

In [21]:
url = "https://search.daum.net/search?w=tot&q=2025%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)
driver.get(url)

NEXT_BTN = "#mor_history_id_0 > div > div.compo-paging > button.btn_next"

# img가 나타날 때까지 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item-thumb c-thumb img"))
)

# 다음 버튼 클릭
driver.find_element(By.CSS_SELECTOR, NEXT_BTN).click()
time.sleep(2)   # 새 썸네일 채워질 시간
print("다음 버튼 클릭 완료")

driver.quit()

다음 버튼 클릭 완료


### 참고. 10년치 썸네일을 파일로 저장하기

In [ ]:
os.makedirs("movie_images", exist_ok=True)
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)

image_urls = []
for year in range(2025, 2015, -1):   # 2025 ~ 2016
    url = ("https://search.daum.net/search?w=tot&q="
           f"{year}%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR")
    driver.get(url)

    # img가 나타날 때까지 대기 (껍데기 아님)
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.item-thumb c-thumb img"))
    )
    soup = BeautifulSoup(driver.page_source, "lxml")
    images = soup.select("div > div.item-thumb > c-thumb > div > a > img")

    # 지연 로딩 placeholder(plazy.svg) 제외하고 실제 이미지만
    real_images = [img for img in images if "plazy" not in img["src"]]

    for rank, image in enumerate(real_images[:10], start=1):
        image_url = image["src"]
        image_urls.append(image_url)

        # 이미지 다운로드는 requests로 (headers는 키워드 인자)
        res = requests.get(image_url, headers=headers)
        res.raise_for_status()
        with open(f"movie_images/{year}_movie_rank_{rank}.jpg", "wb") as fd:
            fd.write(res.content)

driver.quit()
print("저장한 이미지 수:", len(image_urls))

---
## 7. 실전 2: 네이버 증권 (옵션 클릭 후 표 수집)

- 체크박스를 클릭해 표시 항목을 바꾼 뒤 표 수집
- 클릭 후 상태는 정적 크롤링으로 못 만듦 → Selenium으로 가능
- https://finance.naver.com/sise/sise_market_sum.naver

In [ ]:
url = "https://finance.naver.com/sise/sise_market_sum.naver"

options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)
driver.get(url)
time.sleep(1)

# 1) 모든 체크박스 해제
checkboxes = driver.find_elements(By.NAME, "fieldIds")
for cb in checkboxes:
    if cb.is_selected():
        cb.click()
        time.sleep(0.2)

# 2) 원하는 항목만 체크
items_to_select = ["거래량", "시가", "고가", "저가"]
for cb in checkboxes:
                       # 상위디렉토리로 가기
    label = cb.find_element(By.XPATH, "..").find_element(By.TAG_NAME, "label")  # 부모 td의 label
    if label.text in items_to_select:
        cb.click()

# 3) 적용하기 버튼 클릭(copy xpath 활용)
driver.find_element(
    By.XPATH, '//*[@id="contentarea_left"]/div[2]/form/div/div/div/a[1]/img'
).click()
time.sleep(1)

# 셀레늄을 하면 driver.page_source에 들어가있음
# 적용된 표를 pandas로 읽기 (최신 pandas는 StringIO로 감쌈)
tables = pd.read_html(StringIO(driver.page_source))
print("표 개수:", len(tables))

df = tables[1]                              # 시세 표
df = df.dropna(axis="index", how="all")     # 전부 NaN인 행 제거
df = df.dropna(axis="columns", how="all")   # 전부 NaN인 열 제거
driver.quit()

In [34]:
df.head(2)

,N,종목명,현재가,전일비,등락률,액면가,거래량,시가,고가,저가
1,1.0,삼성전자,353000.0,"하락 9,500",-2.62%,100.0,25692117.0,372500.0,374500.0,350000.0
2,2.0,SK하이닉스,2726000.0,"상승 41,000",+1.53%,5000.0,4607117.0,2824000.0,2891000.0,2688000.0
